# Chapter 35 — Planning a Strain Measurement Program

*Companion notebook to Chapter 35 — Planning a Strain Measurement Program.*

A strain measurement program that is poorly planned cannot be rescued by sophisticated post-processing. The decisions made before a single gage is bonded — channel count, sample rate, test duration, anti-aliasing filter settings, shunt calibration procedure — determine whether the data collected will be sufficient to answer the engineering question that motivated the test. They also determine how long it will take to analyze the results, and whether that analysis is even feasible by human effort alone.

This notebook makes those planning decisions quantitative. It calculates the raw data volume for a representative 64-channel, 8-hour acquisition program and converts it to analyst-days — showing immediately that automated analysis is not optional. It then runs through a ten-item pre-test readiness checklist and flags the items most commonly overlooked in the field.

**This notebook demonstrates:**
1. Raw data volume calculation and analyst-day equivalent for a 64-channel, 8-hour test at 2 kHz
2. Pre-test readiness checklist with pass/fail scoring — items commonly missed flagged explicitly

In [ ]:
# Colab: uncomment the next line to install dependencies
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 12,
})

---
## 1 — Data volume: why manual review does not scale

The data-volume calculation is straightforward multiplication — but the result is often surprising to engineers who have not run the numbers before. Sixty-four channels at 2 kHz for 8 hours, stored as 4-byte floats with 20% overhead, produces nearly 18 GB. At a generous review rate of 200 MB per analyst-day, that is roughly 88 analyst-days to fully examine the data by hand. The practical consequence is that no one will ever review this dataset in its entirety: automated feature extraction, alarm detection, and anomaly flagging are not enhancements — they are necessities.

The two helper functions below make the model reusable. Change the channel count, sample rate, or duration to see how quickly a routine test outgrows any hope of manual review.

In [ ]:
# --- Acquisition program parameters (a representative large structural test) ---
N_CHANNELS = 64            # simultaneous strain channels
SAMPLE_RATE_HZ = 2_000     # samples per second, per channel
DURATION_H = 8             # acquisition duration (hours)
BYTES_PER_SAMPLE = 4       # 32-bit float storage
STORAGE_OVERHEAD = 0.20    # headers, timestamps, metadata (fraction)

SECONDS_PER_HOUR = 3_600
BYTES_PER_GB = 1e9
REVIEW_RATE_MB_PER_DAY = 200   # data an analyst can meaningfully review per day


def raw_data_volume_gb(n_channels, sample_rate_hz, duration_h,
                       bytes_per_sample=BYTES_PER_SAMPLE,
                       overhead=STORAGE_OVERHEAD):
    """Raw storage volume in GB for a multi-channel acquisition.

    Every channel is sampled at ``sample_rate_hz`` for ``duration_h`` hours and
    stored as fixed-width samples, inflated by a fractional ``overhead`` for
    headers, timestamps, and metadata.
    """
    n_samples = n_channels * sample_rate_hz * duration_h * SECONDS_PER_HOUR
    total_bytes = n_samples * bytes_per_sample * (1 + overhead)
    return total_bytes / BYTES_PER_GB


def analyst_days(volume_gb, review_rate_mb_per_day=REVIEW_RATE_MB_PER_DAY):
    """Analyst-days needed to review ``volume_gb`` of data by hand."""
    return volume_gb * 1000 / review_rate_mb_per_day


raw_gb = raw_data_volume_gb(N_CHANNELS, SAMPLE_RATE_HZ, DURATION_H)
days = analyst_days(raw_gb)

print(f"Channels: {N_CHANNELS}, fs: {SAMPLE_RATE_HZ} Hz, duration: {DURATION_H} h")
print(f"Raw data volume:            {raw_gb:6.2f} GB")
print(f"Analyst-days to review all: {days:6.1f}  (@ {REVIEW_RATE_MB_PER_DAY} MB/day)")

---
## 2 — Pre-test readiness checklist

The checklist below captures the ten items most commonly audited when a test goes wrong. Three of them — excitation verified **at the bridge** (not at the instrument output), sample rate confirmed against the measurement bandwidth, and anti-alias filter configuration — are the ones most often skipped under time pressure, and most often responsible for data-quality problems discovered only after the structure has been returned to service. Running the checklist before the test costs minutes; discovering the omission afterward can invalidate the entire program.

We store the checklist as an ordered mapping of item to pass/fail state, score it with a small reusable function, and then plot the result so the missing items are impossible to overlook.

In [ ]:
# Pre-test readiness checklist: each item maps to whether it has been completed.
# The three items set to False are the high-risk items most often skipped.
CHECKLIST = {
    "Gage selection documented":     True,
    "Installation inspected":        True,
    "Lead-wire resistance measured": True,
    "Shunt cal before test":         True,
    "Zero reference recorded":       True,
    "Excitation verified at bridge": False,
    "Sample rate >= 10x bandwidth":  False,
    "Anti-alias filter configured":  False,
    "Temperature recorded":          True,
    "Post-test shunt cal":           True,
}


def readiness_score(checklist):
    """Return (passed, total) item counts for a pass/fail checklist."""
    passed = sum(1 for done in checklist.values() if done)
    return passed, len(checklist)


passed, total = readiness_score(CHECKLIST)
print(f"Readiness: {passed}/{total}\n")
for item, done in CHECKLIST.items():
    print(f"  [{'PASS' if done else 'MISS'}] {item}")

In [ ]:
# Visualize the checklist so completed vs. missing items read at a glance.
PASS_COLOR = "#2a9d8f"
MISS_COLOR = "#e76f51"

items = list(CHECKLIST.keys())
done_flags = np.array([CHECKLIST[item] for item in items])
bar_colors = [PASS_COLOR if done else MISS_COLOR for done in done_flags]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
y_pos = np.arange(len(items))
ax.barh(y_pos, np.ones(len(items)), color=bar_colors, edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels(items)
ax.invert_yaxis()             # first checklist item on top
ax.set_xlim(0, 1)
ax.set_xticks([])             # bar length is not meaningful; color carries the status
ax.set_title(f"Pre-test readiness: {passed}/{total} items complete")
ax.legend(handles=[Patch(color=PASS_COLOR, label="complete"),
                   Patch(color=MISS_COLOR, label="missing")],
          loc="lower right", frameon=True)
fig.tight_layout()
plt.show()

## Where to take this next

- **Map the review cliff.** Sweep `SAMPLE_RATE_HZ` and `N_CHANNELS` through `raw_data_volume_gb`, then plot storage (GB) and `analyst_days` as surfaces to find where *your* test crosses from "reviewable by hand" into "must be automated."
- **Weight the checklist by risk.** Replace the pass/fail booleans with weights that penalize the three high-risk items (excitation, sample rate, anti-alias) more heavily, and load the checklist from a versioned YAML template so every project starts from the same baseline.
- **Close the loop to bandwidth.** Add a required measurement bandwidth per channel, compute the minimum anti-alias-safe sample rate (at least 10x bandwidth), and feed that back into `raw_data_volume_gb` so the data-volume estimate and the readiness rule are driven by the same numbers.